# 第3章课堂版：偏好对齐（DPO + GRPO）

这章回答：**模型怎样学会“更偏好好答案”**。

你将看到两种思路：
- DPO：拉大好答案与差答案的分差
- GRPO：在同一组候选里提高高优势回答概率


## 学习目标

- 你能解释 `margin` 的意义。
- 你能解释 `advantage` 的意义。
- 你能通过改奖励或学习率预测结果方向。


In [ ]:
from pathlib import Path
import subprocess, sys, re

ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    if (ROOT.parent / 'projects').exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError(f'未找到项目根目录（应包含 projects/ 目录），当前目录: {Path.cwd()}')

print('使用项目目录:', ROOT)

def run_py(rel_path: str) -> str:
    # 执行项目脚本并返回标准输出文本
    result = subprocess.run(
        [sys.executable, str(ROOT / rel_path)],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(result.stdout)
    return result.stdout


## DPO 实验

关键看：`margin` 是否越来越大。


In [ ]:
out_dpo = run_py('projects/project-02-preference-alignment/dpo_train.py')


In [ ]:
loss_values = [float(x) for x in re.findall(r'平均loss=([0-9.]+)', out_dpo)]
margin_values = [float(x) for x in re.findall(r'margin=([0-9.]+)', out_dpo)]
print('DPO loss 是否下降:', loss_values[-1] < loss_values[0])
print('最终 margin 列表:', margin_values)


## GRPO 实验

关键看：高优势候选概率是否接近 1。


In [ ]:
out_grpo = run_py('projects/project-02-preference-alignment/grpo_train.py')


In [ ]:
prob_values = [float(x) for x in re.findall(r'概率=([0-9.]+)', out_grpo)]
print('候选概率数量:', len(prob_values))
print('最大概率:', max(prob_values))
print('最小概率:', min(prob_values))


## 白话总结

- DPO 的核心是“拉开好坏分差”。
- GRPO 的核心是“同组内相对优势学习”。
- 两者都在做同一件事：让模型更容易选择更优回答。


## 动手练习

1. 在 DPO 脚本里把 `lr` 改成 `0.05` 和 `0.8` 分别试。
2. 在 GRPO 脚本里把一个好候选的奖励降低。
3. 观察 margin/概率如何变化。
